# Análise experimental RFID UHF — gráficos científicos e defensáveis

Este notebook foi reorganizado para produzir figuras simples, lineares e adequadas a relatório técnico ou artigo.

**Princípios adotados**

- uma pergunta experimental por gráfico;
- uma figura por condição principal, evitando excesso de curvas;
- média do RSSI acompanhada por intervalo de confiança de 95%;
- comparação direta com tendências físicas de propagação;
- separação entre distância, potência, tecido, orientação e múltiplas tags;
- nenhuma condição ausente é interpolada ou inventada.

**Figuras produzidas**

1. RSSI médio × distância — uma figura para cada potência;
2. RSSI médio × potência — uma figura para cada distância;
3. atenuação experimental relativa × distância, comparada a referências de 20 e 40 log;
4. potência mínima para inventário completo × distância;
5. comparação pareada sem toalha × com toalha;
6. perda equivalente da toalha × potência;
7. RSSI por orientação e por tag;
8. quantidade de registros por orientação e por tag;
9. ajuste log-distância com resíduos e métricas.

## 1. Enviar CSVs ou ZIP do JRFID APP

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import hashlib

PASTA_DADOS = Path('/content/rfid_dados')
PASTA_SAIDA = Path('/content/rfid_resultados')
PASTA_DADOS.mkdir(exist_ok=True)
PASTA_SAIDA.mkdir(exist_ok=True)

enviados = files.upload()
for nome, conteudo in enviados.items():
    destino = PASTA_DADOS / nome
    destino.write_bytes(conteudo)
    if zipfile.is_zipfile(destino):
        with zipfile.ZipFile(destino, 'r') as z:
            z.extractall(PASTA_DADOS)

arquivos_csv = sorted(PASTA_DADOS.rglob('*.csv'))
print(f'{len(arquivos_csv)} arquivo(s) CSV encontrado(s).')

## 2. Leitura dos CSVs e interpretação do nome dos arquivos

In [ ]:
import csv
import re
import unicodedata
import numpy as np
import pandas as pd

def normalizar(texto):
    texto = unicodedata.normalize('NFKD', str(texto))
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return texto.lower().strip()

def extrair_inteiro(padrao, texto, default=None):
    achado = re.search(padrao, texto)
    return int(achado.group(1)) if achado else default

def metadados_nome(caminho):
    nome_original = caminho.name
    nome = normalizar(nome_original)
    nome = re.sub(r'\(\d+\)(?=\.csv$)', '', nome)

    distancia = extrair_inteiro(r'(\d+)\s*cm', nome)
    potencia = extrair_inteiro(r'(\d+)\s*db', nome)
    n_tags = extrair_inteiro(r'(\d+)\s*tags?', nome, 1)
    n_antenas = extrair_inteiro(r'(\d+)\s*ant', nome, 1)

    if 'deitada' in nome:
        orientacao = '90° horizontal'
    elif re.search(r'90\s*gr', nome):
        orientacao = '90° lateral'
    else:
        angulo = extrair_inteiro(r'(\d+)\s*gr', nome, 0)
        orientacao = f'{angulo}° frontal'

    material = 'toalha seca dobrada' if ('toalha' in nome or 'tolha' in nome) else 'sem toalha'
    return {
        'arquivo': nome_original,
        'distancia_cm': distancia,
        'potencia_nome_db': potencia,
        'n_tags': n_tags,
        'n_antenas': n_antenas,
        'orientacao': orientacao,
        'material': material,
    }

def ler_jrfid(caminho):
    if caminho.stat().st_size < 100:
        return pd.DataFrame()
    with caminho.open('r', encoding='utf-8-sig', errors='replace', newline='') as f:
        linhas = list(csv.reader(f))
    if len(linhas) < 3:
        return pd.DataFrame()
    cabecalho = [x.strip().replace(';;', '') for x in linhas[1]]
    registros=[]
    for linha in linhas[2:]:
        if not linha or not any(str(x).strip() for x in linha):
            continue
        if len(linha)==len(cabecalho)+1:
            cab = cabecalho[:-1] + ['DRM', cabecalho[-1]]
        elif len(linha)==len(cabecalho):
            cab = cabecalho
        else:
            cab = cabecalho + [f'EXTRA_{i}' for i in range(len(linha)-len(cabecalho))]
        registros.append(dict(zip(cab, linha)))
    df = pd.DataFrame(registros)
    if df.empty or 'RSSI' not in df.columns:
        return pd.DataFrame()
    meta = metadados_nome(caminho)
    for k,v in meta.items():
        df[k]=v
    df['RSSI']=pd.to_numeric(df['RSSI'], errors='coerce')
    df['RD ATTEMPTS']=pd.to_numeric(df.get('RD ATTEMPTS'), errors='coerce')
    df['tag_curta']=df['TAG ID'].astype(str).str[-6:]
    df['tid_valido']=~df['TID'].astype(str).str.upper().eq('RDERR')
    if 'FILEDSTRENGTH' in df.columns:
        df['potencia_interna_db']=pd.to_numeric(df['FILEDSTRENGTH'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
    else:
        df['potencia_interna_db']=np.nan
    df['potencia_db']=df['potencia_nome_db'].fillna(df['potencia_interna_db'])
    return df

hashes=set(); tabelas=[]; ignorados=[]
for caminho in arquivos_csv:
    assinatura=hashlib.sha256(caminho.read_bytes()).hexdigest()
    if assinatura in hashes:
        ignorados.append((caminho.name,'duplicado exato')); continue
    hashes.add(assinatura)
    tabela=ler_jrfid(caminho)
    if tabela.empty:
        ignorados.append((caminho.name,'vazio ou sem RSSI'))
    else:
        tabelas.append(tabela)

if not tabelas:
    raise ValueError('Nenhum arquivo válido com RSSI foi encontrado.')

dados=pd.concat(tabelas, ignore_index=True)
dados=dados.dropna(subset=['RSSI','distancia_cm','potencia_db'])
print(f"Arquivos válidos: {dados['arquivo'].nunique()}")
print(f'Registros de RSSI: {len(dados)}')
if ignorados:
    display(pd.DataFrame(ignorados, columns=['arquivo','motivo']))

## 3. Estatísticas: média, desvio-padrão e intervalo de confiança de 95%

In [ ]:
from scipy import stats

grupo_cols=['arquivo','distancia_cm','potencia_db','n_tags','n_antenas','orientacao','material','tag_curta']
resumo_tag=(dados.groupby(grupo_cols, dropna=False)
    .agg(n=('RSSI','count'), media=('RSSI','mean'), desvio=('RSSI','std'), minimo=('RSSI','min'), maximo=('RSSI','max'), tid_validos=('tid_valido','sum'))
    .reset_index())
resumo_tag['erro_padrao']=resumo_tag['desvio']/np.sqrt(resumo_tag['n'])
resumo_tag['t_critico']=resumo_tag['n'].apply(lambda n: stats.t.ppf(0.975,n-1) if n>1 else np.nan)
resumo_tag['ic95']=resumo_tag['erro_padrao']*resumo_tag['t_critico']
resumo_tag['taxa_tid_valido_pct']=100*resumo_tag['tid_validos']/resumo_tag['n']

cond_cols=['arquivo','distancia_cm','potencia_db','n_tags','n_antenas','orientacao','material']
resumo_condicao=(dados.groupby(cond_cols, dropna=False)
    .agg(n=('RSSI','count'), tags_identificadas=('tag_curta','nunique'), media=('RSSI','mean'), desvio=('RSSI','std'), minimo=('RSSI','min'), maximo=('RSSI','max'), tid_validos=('tid_valido','sum'))
    .reset_index())
resumo_condicao['erro_padrao']=resumo_condicao['desvio']/np.sqrt(resumo_condicao['n'])
resumo_condicao['t_critico']=resumo_condicao['n'].apply(lambda n: stats.t.ppf(0.975,n-1) if n>1 else np.nan)
resumo_condicao['ic95']=resumo_condicao['erro_padrao']*resumo_condicao['t_critico']
resumo_condicao['taxa_tid_valido_pct']=100*resumo_condicao['tid_validos']/resumo_condicao['n']
display(resumo_condicao.sort_values(['distancia_cm','potencia_db','material','orientacao']))

## 4. Configuração visual e salvamento

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(8,5.5),'font.size':11,'axes.grid':True,'grid.alpha':0.25,'axes.spines.top':False,'axes.spines.right':False})
def salvar(nome):
    plt.tight_layout()
    caminho=PASTA_SAIDA/nome
    plt.savefig(caminho,dpi=300,bbox_inches='tight')
    plt.show()
    print('Salvo:', caminho)

## 5. RSSI × distância — uma figura para cada potência

In [ ]:
base=resumo_condicao[(resumo_condicao['n_tags']==1)&(resumo_condicao['n_antenas']==1)&(resumo_condicao['orientacao']=='0° frontal')&(resumo_condicao['material']=='sem toalha')].copy()
for potencia,grupo in base.groupby('potencia_db'):
    grupo=grupo.sort_values('distancia_cm')
    if len(grupo)<2: continue
    plt.figure()
    plt.errorbar(grupo['distancia_cm'],grupo['media'],yerr=grupo['ic95'].fillna(0),marker='o',capsize=5,linewidth=1.8)
    plt.xlabel('Distância antena–tag (cm)')
    plt.ylabel('RSSI médio (dBm)')
    plt.title(f'RSSI × distância — FIELDSTRENGTH {int(potencia)} dB')
    plt.figtext(0.5,0.01,'Pontos: média; barras: intervalo de confiança de 95%.',ha='center')
    salvar(f'01_rssi_distancia_{int(potencia)}dB.png')

## 6. RSSI × potência — uma figura para cada distância

In [ ]:
for distancia,grupo in base.groupby('distancia_cm'):
    grupo=grupo.sort_values('potencia_db')
    if len(grupo)<2: continue
    plt.figure()
    plt.errorbar(grupo['potencia_db'],grupo['media'],yerr=grupo['ic95'].fillna(0),marker='o',capsize=5,linewidth=1.8)
    plt.xlabel('FIELDSTRENGTH nominal (dB)')
    plt.ylabel('RSSI médio (dBm)')
    plt.title(f'RSSI × potência — distância de {int(distancia)} cm')
    plt.figtext(0.5,0.01,'Pontos: média; barras: intervalo de confiança de 95%.',ha='center')
    salvar(f'02_rssi_potencia_{int(distancia)}cm.png')

## 7. Atenuação experimental relativa e referências de propagação

Para cada potência, o RSSI é normalizado pelo primeiro ponto de distância disponível:

\[\Delta RSSI(d)=RSSI(d)-RSSI(d_0)\]

São desenhadas duas referências, sem tratá-las como ajuste obrigatório:

- `−20 log10(d/d0)`: referência de potência recebida em enlace simples;
- `−40 log10(d/d0)`: referência idealizada de ida e volta em backscatter.

In [ ]:
for potencia,grupo in base.groupby('potencia_db'):
    grupo=grupo.sort_values('distancia_cm').drop_duplicates('distancia_cm')
    if len(grupo)<3: continue
    d0=grupo['distancia_cm'].iloc[0]; rssi0=grupo['media'].iloc[0]
    d=grupo['distancia_cm'].to_numpy(dtype=float)
    delta=grupo['media'].to_numpy()-rssi0
    ref20=-20*np.log10(d/d0); ref40=-40*np.log10(d/d0)
    plt.figure()
    plt.plot(d,delta,marker='o',linewidth=1.8,label='Medição')
    plt.plot(d,ref20,linestyle='--',label='Referência −20 log10(d/d₀)')
    plt.plot(d,ref40,linestyle=':',label='Referência −40 log10(d/d₀)')
    plt.axhline(0,linewidth=0.8)
    plt.xlabel('Distância antena–tag (cm)')
    plt.ylabel('Variação de RSSI em relação a d₀ (dB)')
    plt.title(f'Atenuação relativa — FIELDSTRENGTH {int(potencia)} dB')
    plt.legend()
    salvar(f'03_atenuacao_relativa_{int(potencia)}dB.png')

## 8. Potência mínima para inventário completo

In [ ]:
base_limite=base.copy(); max_registros=int(base_limite['n'].max())
completos=base_limite[base_limite['n']>=max_registros]
limiar=(completos.groupby('distancia_cm',as_index=False).agg(potencia_minima_db=('potencia_db','min')).sort_values('distancia_cm'))
plt.figure()
plt.step(limiar['distancia_cm'],limiar['potencia_minima_db'],where='mid',linewidth=1.8)
plt.plot(limiar['distancia_cm'],limiar['potencia_minima_db'],marker='o',linestyle='none')
plt.xlabel('Distância antena–tag (cm)')
plt.ylabel('Menor FIELDSTRENGTH com inventário completo (dB)')
plt.title('Limiar experimental de potência para leitura completa')
salvar('04_limiar_potencia_distancia.png')
display(limiar)

## 9. Comparação pareada: sem toalha × com toalha

In [ ]:
toalha=resumo_condicao[(resumo_condicao['distancia_cm']==100)&(resumo_condicao['n_tags']==1)&(resumo_condicao['n_antenas']==1)&(resumo_condicao['orientacao']=='0° frontal')&(resumo_condicao['material'].isin(['sem toalha','toalha seca dobrada']))].copy()
piv_toalha=toalha.pivot_table(index='potencia_db',columns='material',values='media',aggfunc='mean').dropna()
plt.figure()
for potencia,linha in piv_toalha.iterrows():
    plt.plot(['Sem toalha','Com toalha'],[linha['sem toalha'],linha['toalha seca dobrada']],marker='o',linewidth=1.4,label=f'{int(potencia)} dB')
plt.ylabel('RSSI médio (dBm)')
plt.title('Comparação pareada do efeito da toalha — 100 cm')
plt.legend(title='FIELDSTRENGTH')
salvar('05_pareado_sem_com_toalha.png')

## 10. Perda equivalente da toalha × potência

In [ ]:
if not piv_toalha.empty:
    perda=(piv_toalha['sem toalha']-piv_toalha['toalha seca dobrada']).sort_index()
    media_perda=perda.mean(); desvio_perda=perda.std(ddof=1)
    plt.figure()
    plt.plot(perda.index,perda.values,marker='o',linewidth=1.8)
    plt.axhline(media_perda,linestyle='--',label=f'Média = {media_perda:.2f} dB')
    plt.xlabel('FIELDSTRENGTH nominal (dB)')
    plt.ylabel('Perda equivalente do cenário com toalha (dB)')
    plt.title('Perda equivalente da toalha seca dobrada')
    plt.legend()
    salvar('06_perda_equivalente_toalha.png')
    tabela_perda=perda.rename('perda_equivalente_db').reset_index()
    tabela_perda['media_geral_db']=media_perda
    tabela_perda['desvio_entre_potencias_db']=desvio_perda
    display(tabela_perda)

## 11. Duas tags: RSSI por orientação

In [ ]:
orient=resumo_tag[(resumo_tag['distancia_cm']==100)&(resumo_tag['potencia_db']==30)&(resumo_tag['n_tags']==2)&(resumo_tag['n_antenas']==1)].copy()
ordem_orient=['0° frontal','90° horizontal','90° lateral']
orient['orientacao']=pd.Categorical(orient['orientacao'],categories=ordem_orient,ordered=True)
orient=orient.sort_values(['tag_curta','orientacao'])
plt.figure()
for tag,grupo in orient.groupby('tag_curta', observed=True):
    plt.errorbar(grupo['orientacao'].astype(str),grupo['media'],yerr=grupo['ic95'].fillna(0),marker='o',capsize=5,linewidth=1.8,label=tag)
plt.xlabel('Orientação')
plt.ylabel('RSSI médio (dBm)')
plt.title('Influência da orientação — duas tags a 100 cm e 30 dB')
plt.legend(title='Final da tag')
salvar('07_rssi_orientacao_duas_tags.png')

## 12. Duas tags: quantidade de registros por orientação

In [ ]:
plt.figure()
pos=np.arange(len(ordem_orient)); tags=sorted(orient['tag_curta'].dropna().unique()); largura=0.35
for i,tag in enumerate(tags):
    mapa=orient[orient['tag_curta']==tag].set_index('orientacao')['n'].reindex(ordem_orient)
    plt.bar(pos+(i-(len(tags)-1)/2)*largura,mapa.values,largura,label=tag)
plt.xticks(pos,ordem_orient)
plt.xlabel('Orientação')
plt.ylabel('Número de registros')
plt.title('Robustez do inventário por orientação e por tag')
plt.legend(title='Final da tag')
salvar('08_registros_orientacao_duas_tags.png')

## 13. Ajuste log-distância

\[RSSI(d)=A-10n\log_{10}(d)\]

São informados `n`, `R²` e `RMSE`. O ajuste descreve a bancada e não substitui Friis/backscatter.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
ajustes=[]
for potencia,grupo in base.groupby('potencia_db'):
    grupo=grupo.sort_values('distancia_cm').drop_duplicates('distancia_cm')
    if len(grupo)<3: continue
    X=np.log10(grupo[['distancia_cm']].to_numpy(dtype=float)/100.0)
    y=grupo['media'].to_numpy(dtype=float)
    modelo=LinearRegression().fit(X,y); yhat=modelo.predict(X)
    n_exp=-modelo.coef_[0]/10.0; r2=modelo.score(X,y); rmse=np.sqrt(mean_squared_error(y,yhat))
    ajustes.append({'potencia_db':potencia,'A_dBm':modelo.intercept_,'n_experimental':n_exp,'R2':r2,'RMSE_dB':rmse,'numero_pontos':len(grupo)})
    plt.figure()
    plt.plot(grupo['distancia_cm'],y,marker='o',label='Medição')
    plt.plot(grupo['distancia_cm'],yhat,linestyle='--',label='Ajuste log-distância')
    plt.xlabel('Distância antena–tag (cm)')
    plt.ylabel('RSSI médio (dBm)')
    plt.title(f'Ajuste log-distância — FIELDSTRENGTH {int(potencia)} dB')
    plt.legend()
    salvar(f'09_ajuste_log_distancia_{int(potencia)}dB.png')
tabela_ajustes=pd.DataFrame(ajustes)
display(tabela_ajustes)

## 14. Exportar tabelas e figuras

In [ ]:
import shutil
with pd.ExcelWriter(PASTA_SAIDA/'analise_rfid_cientifica.xlsx',engine='openpyxl') as writer:
    dados.to_excel(writer,sheet_name='Dados completos',index=False)
    resumo_tag.to_excel(writer,sheet_name='Resumo por tag',index=False)
    resumo_condicao.to_excel(writer,sheet_name='Resumo por condicao',index=False)
    if 'tabela_perda' in globals(): tabela_perda.to_excel(writer,sheet_name='Perda toalha',index=False)
    if 'tabela_ajustes' in globals(): tabela_ajustes.to_excel(writer,sheet_name='Ajustes log-distancia',index=False)
    if 'limiar' in globals(): limiar.to_excel(writer,sheet_name='Limiar potencia',index=False)

dados.to_csv(PASTA_SAIDA/'dados_consolidados.csv',index=False)
resumo_tag.to_csv(PASTA_SAIDA/'resumo_por_tag.csv',index=False)
resumo_condicao.to_csv(PASTA_SAIDA/'resumo_por_condicao.csv',index=False)
arquivo_zip=shutil.make_archive('/content/resultados_rfid_graficos_cientificos','zip',root_dir=PASTA_SAIDA)
print('Arquivos gerados:')
for item in sorted(PASTA_SAIDA.iterdir()): print('-',item.name)
files.download(arquivo_zip)

## Como defender as figuras no relatório

- **RSSI × distância:** evidencia o decaimento experimental mantendo a potência fixa.
- **RSSI × potência:** mostra a resposta do enlace ao aumento do FIELDSTRENGTH mantendo a geometria fixa.
- **Atenuação relativa:** compara a tendência medida com referências de propagação sem exigir coincidência perfeita.
- **Limiar de potência:** traduz os resultados em requisito operacional.
- **Comparação pareada com toalha:** isola o efeito do material usando condições equivalentes.
- **Orientação por tag:** evita esconder diferenças entre etiquetas ao calcular apenas uma média conjunta.
- **Intervalo de confiança de 95%:** comunica a incerteza da média obtida nas leituras repetidas.
- **Ajuste log-distância:** resume a tendência da bancada por parâmetros mensuráveis, acompanhados por R² e RMSE.